In [160]:
# load packages
import pandas as pd
import csv
import geopandas as gpd
import re
import numpy as np
import censusbatchgeocoder

In [161]:
# import all sheets into a dict with sheet name as key, value as df
sheet_list = ['CIP Sewer Main',
              'CIP Water Main',
              'Breaks Leaks',
              'Homeowner Initiated',
              'Equity',
              'Daycare Private',
              'Daycare Public',
              'Block Level LSLR']

path = '../T126170_Responsive_Document.xlsx'

sheet_dict = {}

for sheet in sheet_list:
    df = pd.read_excel(path, sheet_name=sheet)
    print(sheet)
    print(len(df))
    sheet_dict[sheet] = df

CIP Sewer Main
991
CIP Water Main
1971
Breaks Leaks
3797
Homeowner Initiated
148
Equity
443
Daycare Private
583
Daycare Public
600
Block Level LSLR
1010


In [162]:
display(df)

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed
0,Block Level LSLR,CONSUMER,1037 N DRAKE AVE,In Progress,60651,36,NaT
1,Block Level LSLR,CONSUMER,1633 43 W 76TH ST,In Progress-New Record,60620,17,NaT
2,Block Level LSLR,PUBLIC,1633 43 W 76TH ST,In Progress-New Record,60620,17,NaT
3,Block Level LSLR,CONSUMER,2300 W 59TH ST,Completed,60636,16,2025-12-03 00:00:00
4,Block Level LSLR,PUBLIC,2300 W 59TH ST,Completed,60636,16,2025-09-25 11:40:00
...,...,...,...,...,...,...,...
1005,Block Level LSLR,PUBLIC,8155 S JEFFERY BLVD,Completed,60617,8,2025-11-13 00:00:00
1006,Block Level LSLR,CONSUMER,8158 S JEFFERY BLVD,Completed,60617,8,2025-11-19 00:00:00
1007,Block Level LSLR,PUBLIC,8158 S JEFFERY BLVD,Completed,60617,8,2025-11-19 00:00:00
1008,Block Level LSLR,CONSUMER,8159 S JEFFERY BLVD,Completed,60617,8,2025-11-13 00:00:00


In [163]:
programs = df

In [164]:
# convert date and add new date cols
programs['Date Completed'] = pd.to_datetime(programs['Date Completed'])
programs['completed_year'] = programs['Date Completed'].dt.year
programs['completed_month_year'] = programs['Date Completed'].dt.strftime('%m/%Y')
programs['completed_day'] = programs['Date Completed'].dt.strftime('%m-%d')

In [165]:
# first and last dates completed
print(programs[programs['Status'] == 'Completed'].sort_values('completed_day').head(2))
print(programs[programs['Status'] == 'Completed'].sort_values('completed_day').tail(2))

         LSLR Program Public or Consumer             Work Address     Status  \
103  Block Level LSLR           CONSUMER  3101 N NARRAGANSETT AVE  Completed   
104  Block Level LSLR             PUBLIC  3101 N NARRAGANSETT AVE  Completed   

       ZIP  Ward      Date Completed  completed_year completed_month_year  \
103  60634    30 2025-09-02 00:00:00          2025.0              09/2025   
104  60634    30 2025-09-02 14:52:00          2025.0              09/2025   

    completed_day  
103         09-02  
104         09-02  
         LSLR Program Public or Consumer         Work Address     Status  \
71   Block Level LSLR           CONSUMER    2500 10 W 60TH ST  Completed   
482  Block Level LSLR           CONSUMER  6011 S CAMPBELL AVE  Completed   

       ZIP  Ward      Date Completed  completed_year completed_month_year  \
71   60629    16 2025-12-19 00:00:00          2025.0              12/2025   
482  60629    16 2025-12-19 12:13:00          2025.0              12/2025   

    co

In [166]:
programs.to_csv('../processed/replacements_unduplicated_uncleaned_update.csv', index=False)

In [167]:
programs[programs['Work Address'] == '4323 W Cermak RD']

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day


In [168]:
programs[programs['Work Address'].str.contains('2500')]

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day
71,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19,2025.0,12/2025,12-19
72,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19,2025.0,12/2025,12-19


In [169]:
# okay so she was just gutchecking I think

In [170]:
print(programs['Work Address'].dtype)

str


In [171]:
programs[programs['Work Address'].str.contains('02')]

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day
82,Block Level LSLR,CONSUMER,3022 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
83,Block Level LSLR,PUBLIC,3022 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
84,Block Level LSLR,CONSUMER,3023 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
85,Block Level LSLR,PUBLIC,3023 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
86,Block Level LSLR,CONSUMER,3026 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
87,Block Level LSLR,PUBLIC,3026 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
88,Block Level LSLR,CONSUMER,3027 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
89,Block Level LSLR,PUBLIC,3027 S UNION AVE,Completed,60616,11,2025-10-20 00:00:00,2025.0,10/2025,10-20
90,Block Level LSLR,CONSUMER,3028 S UNION AVE,Completed,60616,11,2025-10-15 00:00:00,2025.0,10/2025,10-15
91,Block Level LSLR,PUBLIC,3028 S UNION AVE,Completed,60616,11,2025-10-15 00:00:00,2025.0,10/2025,10-15


In [172]:
# yeahhh we are fine here

## excluding noncomplete records

In [173]:
programs.groupby('Status', dropna=False).size()

Status
Completed                 568
In Progress                36
In Progress-New Record    406
dtype: int64

In [174]:
step1 = programs[(programs['Status'] == 'Completed') & (programs['Date Completed'].notnull())].copy()

print(len(step1))

568


In [175]:
print(len(programs) - len(step1))

442


## cleaning addresses

In [176]:
# standardize addresses
def clean_address(address):
    clean_address = address.replace(',','')
    clean_address = clean_address.replace('.','')
    clean_address = clean_address.replace('-',' ')
    clean_address = clean_address.upper()
    
    return clean_address

step2 = step1.copy()

step2['clean_address'] = step2['Work Address'].apply(clean_address)
step2.head()

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address
3,Block Level LSLR,CONSUMER,2300 W 59TH ST,Completed,60636,16,2025-12-03 00:00:00,2025.0,12/2025,12-03,2300 W 59TH ST
4,Block Level LSLR,PUBLIC,2300 W 59TH ST,Completed,60636,16,2025-09-25 11:40:00,2025.0,09/2025,09-25,2300 W 59TH ST
5,Block Level LSLR,PUBLIC,2301 W GARFIELD BLVD,Completed,60636,15,2025-09-25 13:54:00,2025.0,09/2025,09-25,2301 W GARFIELD BLVD
71,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 10 W 60TH ST
72,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 10 W 60TH ST


In [177]:
# use only first street number if range
step2['clean_address'] = step2['clean_address'].str.replace(r"^(\d+)\s+\d+\b", r"\1", regex=True)

In [178]:
display(step2[['clean_address','Work Address']])

,clean_address,Work Address
3,2300 W 59TH ST,2300 W 59TH ST
4,2300 W 59TH ST,2300 W 59TH ST
5,2301 W GARFIELD BLVD,2301 W GARFIELD BLVD
71,2500 W 60TH ST,2500 10 W 60TH ST
72,2500 W 60TH ST,2500 10 W 60TH ST
...,...,...
1005,8155 S JEFFERY BLVD,8155 S JEFFERY BLVD
1006,8158 S JEFFERY BLVD,8158 S JEFFERY BLVD
1007,8158 S JEFFERY BLVD,8158 S JEFFERY BLVD
1008,8159 S JEFFERY BLVD,8159 S JEFFERY BLVD


In [179]:
# spot check
step2[step2['Work Address'].str.contains('2043 n. Humboldt blvd.')]

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address


In [180]:
# grabbing a diff example — 1040 N Marshfield ave
# spot check

step2[step2['Work Address'].str.contains('2434 w. BERENICE')]


,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address


In [181]:
# skip fo rnow? 
# 5:44 pm 6/9/26

# may need to standardize additional entries/outliers here

In [182]:
step2[step2['Work Address'].str.contains('8147 49 S JEFFERY BLVD')]

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address
994,Block Level LSLR,PUBLIC,8147 49 S JEFFERY BLVD,Completed,60617,8,2025-11-12,2025.0,11/2025,11-12,8147 S JEFFERY BLVD
995,Block Level LSLR,CONSUMER,8147 49 S JEFFERY BLVD,Completed,60617,8,2025-11-12,2025.0,11/2025,11-12,8147 S JEFFERY BLVD


In [183]:
# okay that one worked!


## remove dupes at same line section

very clever!!

In [184]:
step2[step2['Work Address'].str.contains('8151 S JEFFERY BLVD')]

# swapping with new comparable example

,LSLR Program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address
998,Block Level LSLR,CONSUMER,8151 S JEFFERY BLVD,Completed,60617,8,2025-12-02,2025.0,12/2025,12-02,8151 S JEFFERY BLVD
999,Block Level LSLR,PUBLIC,8151 S JEFFERY BLVD,Completed,60617,8,2025-12-02,2025.0,12/2025,12-02,8151 S JEFFERY BLVD


In [185]:
# i think a missing step here with "program"

step2.rename(columns={
    "LSLR Program":"program"
}, inplace=True)
display(step2)

step3 = step2.copy()


,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address
3,Block Level LSLR,CONSUMER,2300 W 59TH ST,Completed,60636,16,2025-12-03 00:00:00,2025.0,12/2025,12-03,2300 W 59TH ST
4,Block Level LSLR,PUBLIC,2300 W 59TH ST,Completed,60636,16,2025-09-25 11:40:00,2025.0,09/2025,09-25,2300 W 59TH ST
5,Block Level LSLR,PUBLIC,2301 W GARFIELD BLVD,Completed,60636,15,2025-09-25 13:54:00,2025.0,09/2025,09-25,2301 W GARFIELD BLVD
71,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 W 60TH ST
72,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 W 60TH ST
...,...,...,...,...,...,...,...,...,...,...,...
1005,Block Level LSLR,PUBLIC,8155 S JEFFERY BLVD,Completed,60617,8,2025-11-13 00:00:00,2025.0,11/2025,11-13,8155 S JEFFERY BLVD
1006,Block Level LSLR,CONSUMER,8158 S JEFFERY BLVD,Completed,60617,8,2025-11-19 00:00:00,2025.0,11/2025,11-19,8158 S JEFFERY BLVD
1007,Block Level LSLR,PUBLIC,8158 S JEFFERY BLVD,Completed,60617,8,2025-11-19 00:00:00,2025.0,11/2025,11-19,8158 S JEFFERY BLVD
1008,Block Level LSLR,CONSUMER,8159 S JEFFERY BLVD,Completed,60617,8,2025-11-13 00:00:00,2025.0,11/2025,11-13,8159 S JEFFERY BLVD


In [186]:
display(step2.head(3))
display(step3.head(3))

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address
3,Block Level LSLR,CONSUMER,2300 W 59TH ST,Completed,60636,16,2025-12-03 00:00:00,2025.0,12/2025,12-03,2300 W 59TH ST
4,Block Level LSLR,PUBLIC,2300 W 59TH ST,Completed,60636,16,2025-09-25 11:40:00,2025.0,09/2025,09-25,2300 W 59TH ST
5,Block Level LSLR,PUBLIC,2301 W GARFIELD BLVD,Completed,60636,15,2025-09-25 13:54:00,2025.0,09/2025,09-25,2301 W GARFIELD BLVD


,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address
3,Block Level LSLR,CONSUMER,2300 W 59TH ST,Completed,60636,16,2025-12-03 00:00:00,2025.0,12/2025,12-03,2300 W 59TH ST
4,Block Level LSLR,PUBLIC,2300 W 59TH ST,Completed,60636,16,2025-09-25 11:40:00,2025.0,09/2025,09-25,2300 W 59TH ST
5,Block Level LSLR,PUBLIC,2301 W GARFIELD BLVD,Completed,60636,15,2025-09-25 13:54:00,2025.0,09/2025,09-25,2301 W GARFIELD BLVD


In [187]:
# sort by date completed and drop duplicates that are the same accross public/consumer, address and program

step3 = step3.sort_values('Date Completed', ascending=False).drop_duplicates(subset=['clean_address', 'program', 'Public or Consumer'], keep='first')

print(len(step3))
print(len(step2) - len(step3))

568
0


In [188]:
# are they all unique?

In [189]:
step3[step3['Work Address'].str.contains('8141 S JEFFERY BLVD')]

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address
988,Block Level LSLR,PUBLIC,8141 S JEFFERY BLVD,Completed,60617,8,2025-11-21,2025.0,11/2025,11-21,8141 S JEFFERY BLVD
989,Block Level LSLR,CONSUMER,8141 S JEFFERY BLVD,Completed,60617,8,2025-11-21,2025.0,11/2025,11-21,8141 S JEFFERY BLVD


## count only records with both line sections completeted

In [190]:
step3.groupby('Public or Consumer', dropna=False).size()

Public or Consumer
CONSUMER    283
PUBLIC      285
dtype: int64

In [191]:
# standardize daycare label
step3['Public or Consumer'] = np.where(step3['Public or Consumer'] == 'CONSUMER - DAYCARE', 'CONSUMER', step3['Public or Consumer'])

In [192]:
# add the 1-based index column for every record to count by
step3['id'] = range(1, len(step3) + 1)

In [193]:
# add program address identifier
step3['program_clean_address'] = step3['program'] + ' ' + step3['clean_address']

In [194]:
# are there duplicate addresses accross programs? yes
step3[step3['clean_address'] == '8100 S JEFFERY BLVD']

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address,id,program_clean_address
938,Block Level LSLR,PUBLIC,8100 S JEFFERY BLVD,Completed,60617,8,2025-11-04,2025.0,11/2025,11-04,8100 S JEFFERY BLVD,239,Block Level LSLR 8100 S JEFFERY BLVD
939,Block Level LSLR,CONSUMER,8100 S JEFFERY BLVD,Completed,60617,8,2025-11-04,2025.0,11/2025,11-04,8100 S JEFFERY BLVD,241,Block Level LSLR 8100 S JEFFERY BLVD


In [195]:
# skip for now because these are the same program

In [196]:
completed_line_sections_test = pd.pivot_table(step3,
              index='clean_address',
              columns='Public or Consumer',
              values='id',
              aggfunc='count').reset_index()

In [197]:
# exclude addresses that just have consumer OR public side AND do not have both as required
# addresses_to_exclude = completed_line_sections_test[((completed_line_sections_test['CONSUMER'].isnull()) | (completed_line_sections_test['PUBLIC'].isnull())) & (completed_line_sections_test['BOTH AS REQUIRED'].isnull())].copy()


# cam add — there are no "BOTH AS REQUIRED" lines. I'm going to remove it from the code below. (5:54 pm 6/9/26)
# exclude addresses that just have consumer OR public side AND do not have both as required
addresses_to_exclude = completed_line_sections_test[((completed_line_sections_test['CONSUMER'].isnull()) | (completed_line_sections_test['PUBLIC'].isnull()))].copy()

In [198]:
print(len(addresses_to_exclude))

6


In [199]:
# okay! only excluding 6 addresses
display(addresses_to_exclude)

Public or Consumer,clean_address,CONSUMER,PUBLIC
1,2301 W GARFIELD BLVD,NaN,1.0
157,5930 S CAMPBELL AVE,NaN,1.0
159,5933 S CAMPBELL AVE,NaN,1.0
172,5954 S CAMPBELL AVE,NaN,1.0
180,6230 W BARRY AVE,1.0,NaN
182,6252 W BARRY AVE,1.0,NaN


In [200]:
# create list of addresses and filter them out
addresses_to_exclude_list = addresses_to_exclude['clean_address'].to_list()

step4 = step3[~step3['clean_address'].isin(addresses_to_exclude_list)].copy()

In [201]:
# check
print(len(step3))
print(len(step4))
print(len(step3) - len(step4))

568
562
6


In [202]:
# slay

In [203]:
public_set = []
consumer_set = []

# check again that the same addresses in public are in private
public_set = set(step4[step4['Public or Consumer'] == 'PUBLIC']['clean_address'].to_list())
consumer_set = set(step4[step4['Public or Consumer'] == 'CONSUMER']['clean_address'].to_list())

display(public_set)
display(consumer_set)

{'2300 W 59TH ST',
 '2500 W 60TH ST',
 '2501 W 59TH ST',
 '3010 S UNION AVE',
 '3011 S UNION AVE',
 '3022 S UNION AVE',
 '3023 S UNION AVE',
 '3026 S UNION AVE',
 '3027 S UNION AVE',
 '3028 S UNION AVE',
 '3029 S UNION AVE',
 '3039 S UNION AVE',
 '3042 S UNION AVE',
 '3043 S UNION AVE',
 '3057 N NARRAGANSETT AVE',
 '3101 N NARRAGANSETT AVE',
 '4501 S LAMON AVE',
 '4505 S LAMON AVE',
 '4507 S LAMON AVE',
 '4511 S LAMON AVE',
 '4515 S LAMON AVE',
 '4517 S LAMON AVE',
 '4519 S LAMON AVE',
 '4521 S LAMON AVE',
 '4525 S LAMON AVE',
 '4529 S LAMON AVE',
 '4531 S LAMON AVE',
 '4535 S LAMON AVE',
 '4537 S LAMON AVE',
 '4541 S LAMON AVE',
 '4543 S LAMON AVE',
 '4547 S LAMON AVE',
 '4549 S LAMON AVE',
 '4553 S LAMON AVE',
 '4555 S LAMON AVE',
 '4559 S LAMON AVE',
 '4601 S LAMON AVE',
 '4605 S LAMON AVE',
 '4609 S LAMON AVE',
 '4611 S LAMON AVE',
 '4615 S LAMON AVE',
 '4621 S LAMON AVE',
 '4623 S LAMON AVE',
 '4627 S LAMON AVE',
 '4631 S LAMON AVE',
 '4633 S LAMON AVE',
 '4635 S LAMON AVE',
 '463

{'2300 W 59TH ST',
 '2500 W 60TH ST',
 '2501 W 59TH ST',
 '3010 S UNION AVE',
 '3011 S UNION AVE',
 '3022 S UNION AVE',
 '3023 S UNION AVE',
 '3026 S UNION AVE',
 '3027 S UNION AVE',
 '3028 S UNION AVE',
 '3029 S UNION AVE',
 '3039 S UNION AVE',
 '3042 S UNION AVE',
 '3043 S UNION AVE',
 '3057 N NARRAGANSETT AVE',
 '3101 N NARRAGANSETT AVE',
 '4501 S LAMON AVE',
 '4505 S LAMON AVE',
 '4507 S LAMON AVE',
 '4511 S LAMON AVE',
 '4515 S LAMON AVE',
 '4517 S LAMON AVE',
 '4519 S LAMON AVE',
 '4521 S LAMON AVE',
 '4525 S LAMON AVE',
 '4529 S LAMON AVE',
 '4531 S LAMON AVE',
 '4535 S LAMON AVE',
 '4537 S LAMON AVE',
 '4541 S LAMON AVE',
 '4543 S LAMON AVE',
 '4547 S LAMON AVE',
 '4549 S LAMON AVE',
 '4553 S LAMON AVE',
 '4555 S LAMON AVE',
 '4559 S LAMON AVE',
 '4601 S LAMON AVE',
 '4605 S LAMON AVE',
 '4609 S LAMON AVE',
 '4611 S LAMON AVE',
 '4615 S LAMON AVE',
 '4621 S LAMON AVE',
 '4623 S LAMON AVE',
 '4627 S LAMON AVE',
 '4631 S LAMON AVE',
 '4633 S LAMON AVE',
 '4635 S LAMON AVE',
 '463

In [204]:
diff_set = consumer_set - public_set

# cullerton is both as required so we're good
diff_set

set()

In [205]:
# empty set?

## geocode

Let's try and geocode against the existing dataset?

In [206]:
existing = []
existing = pd.read_csv('../processed/replacements_cleaned_updated.csv')
display(existing)

,Public or Consumer,Work Address,Status,Zip,Ward,Date Completed,program,completed_year,completed_month_year,completed_day,...,geoid,matched_address,m_is_intersection,_merge,geometry,GEOID,NAME,Community Area,CA,completed_day_full
0,PUBLIC,11014 S AVENUE H,Completed,60617,10.0,2025-08-25 15:00:00,Breaks Leaks,2025,08/2025,08-25,...,NaN,NaN,NaN,left_only,POINT (-87.532946029631 41.695162818406),17031520500,5205.00,East Side,52,2025-08-25
1,BOTH AS REQUIRED,1127 W Wolfram,Completed,60657,44.0,2025-08-22 17:09:00,Homeowner Initiated,2025,08/2025,08-22,...,NaN,NaN,NaN,left_only,POINT (-87.657363092128 41.933434330781),17031062900,629.00,Lake View,6,2025-08-22
2,CONSUMER,35 37 W 113TH ST,Completed,60628,9.0,2025-08-11 00:00:00,Block Level LSLR,2025,08/2025,08-11,...,NaN,NaN,NaN,left_only,POINT (-87.624322088096 41.688880284153),17031491300,4913.00,Roseland,49,2025-08-11
3,CONSUMER,3650 N NOTTINGHAM AVE,Completed,60634,38.0,2025-08-04 16:56:00,Breaks Leaks,2025,08/2025,08-04,...,NaN,NaN,NaN,left_only,POINT (-87.804657092815 41.946235762097),17031170300,1703.00,Dunning,17,2025-08-04
4,BOTH AS REQUIRED,2221 W. WIINNEMAC,Completed,60625,40.0,2025-07-30 00:00:00,Homeowner Initiated,2025,07/2025,07-30,...,NaN,NaN,NaN,left_only,POINT (-87.685162110066 41.973043813187),17031040401,404.01,Lincoln Square,4,2025-07-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12647,BOTH AS REQUIRED,7743 S Bishop St,Completed,60620,17.0,2021-09-07 00:00:00,Equity,2021,09/2021,09-07,...,1.703171e+10,"7743 S BISHOP ST, CHICAGO IL 60620",False,both,POINT (-87.660182033758 41.7528075674),17031710300,7103.00,Auburn Gresham,71,2021-09-07
12648,BOTH AS REQUIRED,8343 S HERMITAGE AVE,Completed,60620,18.0,2021-08-30 00:00:00,Equity,2021,08/2021,08-30,...,1.703171e+10,"8343 S HERMITAGE AVE, CHICAGO IL 60620",False,both,POINT (-87.666791055855 41.741974851484),17031711200,7112.00,Auburn Gresham,71,2021-08-30
12649,BOTH AS REQUIRED,8338 S Perry Ave,Completed,60620,6.0,2021-07-30 00:00:00,Equity,2021,07/2021,07-30,...,1.703184e+10,"8338 S PERRY AVE, CHICAGO IL 60620",False,both,POINT (-87.626889012206 41.742621018292),17031842400,8424.00,Chatham,44,2021-07-30
12650,BOTH AS REQUIRED,217 S Bell Ave,Completed,60612,27.0,2021-06-15 00:00:00,Homeowner Initiated,2021,06/2021,06-15,...,1.703184e+10,"217 S BELL AVE, CHICAGO IL 60612",False,both,POINT (-87.682664156211 41.878518344119),17031838000,8380.00,Near West Side,28,2021-06-15


In [207]:
step5 = step4.copy()

In [208]:
step5[step5['Work Address'] == '8141 S JEFFERY BLVD']

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address,id,program_clean_address
988,Block Level LSLR,PUBLIC,8141 S JEFFERY BLVD,Completed,60617,8,2025-11-21,2025.0,11/2025,11-21,8141 S JEFFERY BLVD,153,Block Level LSLR 8141 S JEFFERY BLVD
989,Block Level LSLR,CONSUMER,8141 S JEFFERY BLVD,Completed,60617,8,2025-11-21,2025.0,11/2025,11-21,8141 S JEFFERY BLVD,158,Block Level LSLR 8141 S JEFFERY BLVD


In [209]:
# "Zip" changed to "ZIP" in code to match

# create a join field in step 5
step5['full_address'] = step5['clean_address'] + ', CHICAGO IL' + ' ' + step5['ZIP'].astype(str)
step5.head(2)

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address,id,program_clean_address,full_address
482,Block Level LSLR,CONSUMER,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:13:00,2025.0,12/2025,12-19,6011 S CAMPBELL AVE,1,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO IL 60629"
483,Block Level LSLR,PUBLIC,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:10:00,2025.0,12/2025,12-19,6011 S CAMPBELL AVE,2,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO IL 60629"


In [210]:
len(step5)

562

In [211]:
print(step5.columns)

Index(['program', 'Public or Consumer', 'Work Address', 'Status', 'ZIP',
       'Ward', 'Date Completed', 'completed_year', 'completed_month_year',
       'completed_day', 'clean_address', 'id', 'program_clean_address',
       'full_address'],
      dtype='str')


In [212]:
print(existing.columns)

Index(['Public or Consumer', 'Work Address', 'Status', 'Zip', 'Ward',
       'Date Completed', 'program', 'completed_year', 'completed_month_year',
       'completed_day', 'clean_address', 'program_clean_address',
       'full_address', 'row', 'classification_for_entire_service_line',
       'stnum1', 'stnum2', 'stdir', 'stname', 'sttype', 'zip', 'geocoder',
       'lat', 'long', 'geoid', 'matched_address', 'm_is_intersection',
       '_merge', 'geometry', 'GEOID', 'NAME', 'Community Area', 'CA',
       'completed_day_full'],
      dtype='str')


In [213]:
existing.rename(columns={"_merge":"_merge_existing"},inplace=True)

In [214]:
# left merge
merged = []
merged = pd.merge(step5, existing, left_on='full_address', right_on='matched_address', how='left', indicator=True)

In [215]:
merged['_merge'].value_counts()

_merge
left_only     556
both            6
right_only      0
Name: count, dtype: int64

In [216]:
len(merged)

562

In [217]:
display(merged)

,program_x,Public or Consumer_x,Work Address_x,Status_x,ZIP,Ward_x,Date Completed_x,completed_year_x,completed_month_year_x,completed_day_x,...,matched_address,m_is_intersection,_merge_existing,geometry,GEOID,NAME,Community Area,CA,completed_day_full,_merge
0,Block Level LSLR,CONSUMER,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:13:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Block Level LSLR,PUBLIC,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:10:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Block Level LSLR,CONSUMER,6000 S CAMPBELL AVE,Completed,60629,16,2025-12-18 00:00:00,2025.0,12/2025,12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,Block Level LSLR,CONSUMER,6330 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
558,Block Level LSLR,CONSUMER,6334 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
559,Block Level LSLR,CONSUMER,6340 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
560,Block Level LSLR,PUBLIC,3101 N NARRAGANSETT AVE,Completed,60634,30,2025-09-02 14:52:00,2025.0,09/2025,09-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [218]:
print(merged.columns)

Index(['program_x', 'Public or Consumer_x', 'Work Address_x', 'Status_x',
       'ZIP', 'Ward_x', 'Date Completed_x', 'completed_year_x',
       'completed_month_year_x', 'completed_day_x', 'clean_address_x', 'id',
       'program_clean_address_x', 'full_address_x', 'Public or Consumer_y',
       'Work Address_y', 'Status_y', 'Zip', 'Ward_y', 'Date Completed_y',
       'program_y', 'completed_year_y', 'completed_month_year_y',
       'completed_day_y', 'clean_address_y', 'program_clean_address_y',
       'full_address_y', 'row', 'classification_for_entire_service_line',
       'stnum1', 'stnum2', 'stdir', 'stname', 'sttype', 'zip', 'geocoder',
       'lat', 'long', 'geoid', 'matched_address', 'm_is_intersection',
       '_merge_existing', 'geometry', 'GEOID', 'NAME', 'Community Area', 'CA',
       'completed_day_full', '_merge'],
      dtype='str')


In [219]:
#non-exact with existing data
# non-exact matches with peter's data

merged[merged['_merge'] == 'left_only'][['clean_address_x', 'full_address_x', 'clean_address_y', 'full_address_y']]

,clean_address_x,full_address_x,clean_address_y,full_address_y
0,6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO IL 60629",NaN,NaN
1,6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO IL 60629",NaN,NaN
2,2500 W 60TH ST,"2500 W 60TH ST, CHICAGO IL 60629",NaN,NaN
3,2500 W 60TH ST,"2500 W 60TH ST, CHICAGO IL 60629",NaN,NaN
4,6000 S CAMPBELL AVE,"6000 S CAMPBELL AVE, CHICAGO IL 60629",NaN,NaN
...,...,...,...,...
557,6330 W BARRY AVE,"6330 W BARRY AVE, CHICAGO IL 60634",NaN,NaN
558,6334 W BARRY AVE,"6334 W BARRY AVE, CHICAGO IL 60634",NaN,NaN
559,6340 W BARRY AVE,"6340 W BARRY AVE, CHICAGO IL 60634",NaN,NaN
560,3101 N NARRAGANSETT AVE,"3101 N NARRAGANSETT AVE, CHICAGO IL 60634",NaN,NaN


In [220]:
# yeah I think I have to scrap this part in favor of skipping to geocoding fr fr

In [221]:
# sorry amy!

In [222]:
# geocode 1,085 with census geocoder
to_geocode = merged[merged['_merge'] == 'left_only'].copy()
# prep data for geocoding
to_geocode['city'] = 'Chicago'
to_geocode['state'] = 'IL'

to_geocode_formatted = to_geocode[['id','clean_address_x','city','state','ZIP']].copy()
to_geocode_formatted.columns = ['id','address','city','state','zipcode']
to_geocode_formatted

,id,address,city,state,zipcode
0,1,6011 S CAMPBELL AVE,Chicago,IL,60629
1,2,6011 S CAMPBELL AVE,Chicago,IL,60629
2,3,2500 W 60TH ST,Chicago,IL,60629
3,4,2500 W 60TH ST,Chicago,IL,60629
4,5,6000 S CAMPBELL AVE,Chicago,IL,60629
...,...,...,...,...,...
557,564,6330 W BARRY AVE,Chicago,IL,60634
558,565,6334 W BARRY AVE,Chicago,IL,60634
559,566,6340 W BARRY AVE,Chicago,IL,60634
560,567,3101 N NARRAGANSETT AVE,Chicago,IL,60634


In [223]:
# run thru batch geocoder
result = censusbatchgeocoder.geocode(to_geocode_formatted.to_dict("records"))

In [224]:
results_df = pd.DataFrame(result)
results_df.head()

,id,address,city,state,zipcode,geocoded_address,is_match,is_exact,returned_address,coordinates,tiger_line,side,state_fips,county_fips,tract,block,longitude,latitude
0,1,6011 S CAMPBELL AVE,Chicago,IL,60629,"6011 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6011 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686151139103,41.784467129116",111804823,L,17,031,835000,1010,-87.686151,41.784467
1,2,6011 S CAMPBELL AVE,Chicago,IL,60629,"6011 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6011 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686151139103,41.784467129116",111804823,L,17,031,835000,1010,-87.686151,41.784467
2,3,2500 W 60TH ST,Chicago,IL,60629,"2500 W 60TH ST, Chicago, IL, 60629",Match,Exact,"2500 W 60TH ST, CHICAGO, IL, 60629","-87.686310582357,41.78475446124",605308184,R,17,031,835000,1006,-87.686311,41.784754
3,4,2500 W 60TH ST,Chicago,IL,60629,"2500 W 60TH ST, Chicago, IL, 60629",Match,Exact,"2500 W 60TH ST, CHICAGO, IL, 60629","-87.686310582357,41.78475446124",605308184,R,17,031,835000,1006,-87.686311,41.784754
4,5,6000 S CAMPBELL AVE,Chicago,IL,60629,"6000 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6000 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686309136878,41.784638811931",111804823,R,17,031,835000,1009,-87.686309,41.784639


In [225]:
len(results_df[results_df['longitude'].isnull()])

0

In [226]:
# SLAYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYYY this rules

In [227]:
# skipping manual geocoding step

In [228]:
results_df.to_csv('./newdata_geocoded.csv', index=False)

In [229]:
step6 = pd.read_csv('./newdata_geocoded.csv')

In [230]:
step6.head(2)

,id,address,city,state,zipcode,geocoded_address,is_match,is_exact,returned_address,coordinates,tiger_line,side,state_fips,county_fips,tract,block,longitude,latitude
0,1,6011 S CAMPBELL AVE,Chicago,IL,60629,"6011 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6011 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686151139103,41.784467129116",111804823,L,17,31,835000,1010,-87.686151,41.784467
1,2,6011 S CAMPBELL AVE,Chicago,IL,60629,"6011 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6011 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686151139103,41.784467129116",111804823,L,17,31,835000,1010,-87.686151,41.784467


## dedupe

In [231]:
# i think this has to be joined back in maybe?

In [232]:
display(merged)

,program_x,Public or Consumer_x,Work Address_x,Status_x,ZIP,Ward_x,Date Completed_x,completed_year_x,completed_month_year_x,completed_day_x,...,matched_address,m_is_intersection,_merge_existing,geometry,GEOID,NAME,Community Area,CA,completed_day_full,_merge
0,Block Level LSLR,CONSUMER,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:13:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,Block Level LSLR,PUBLIC,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:10:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Block Level LSLR,CONSUMER,6000 S CAMPBELL AVE,Completed,60629,16,2025-12-18 00:00:00,2025.0,12/2025,12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,Block Level LSLR,CONSUMER,6330 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
558,Block Level LSLR,CONSUMER,6334 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
559,Block Level LSLR,CONSUMER,6340 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
560,Block Level LSLR,PUBLIC,3101 N NARRAGANSETT AVE,Completed,60634,30,2025-09-02 14:52:00,2025.0,09/2025,09-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [233]:
display(step5)

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address,id,program_clean_address,full_address
482,Block Level LSLR,CONSUMER,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:13:00,2025.0,12/2025,12-19,6011 S CAMPBELL AVE,1,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO IL 60629"
483,Block Level LSLR,PUBLIC,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:10:00,2025.0,12/2025,12-19,6011 S CAMPBELL AVE,2,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO IL 60629"
71,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 W 60TH ST,3,Block Level LSLR 2500 W 60TH ST,"2500 W 60TH ST, CHICAGO IL 60629"
72,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 W 60TH ST,4,Block Level LSLR 2500 W 60TH ST,"2500 W 60TH ST, CHICAGO IL 60629"
474,Block Level LSLR,CONSUMER,6000 S CAMPBELL AVE,Completed,60629,16,2025-12-18 00:00:00,2025.0,12/2025,12-18,6000 S CAMPBELL AVE,5,Block Level LSLR 6000 S CAMPBELL AVE,"6000 S CAMPBELL AVE, CHICAGO IL 60629"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,Block Level LSLR,CONSUMER,6330 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,6330 W BARRY AVE,564,Block Level LSLR 6330 W BARRY AVE,"6330 W BARRY AVE, CHICAGO IL 60634"
555,Block Level LSLR,CONSUMER,6334 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,6334 W BARRY AVE,565,Block Level LSLR 6334 W BARRY AVE,"6334 W BARRY AVE, CHICAGO IL 60634"
559,Block Level LSLR,CONSUMER,6340 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,6340 W BARRY AVE,566,Block Level LSLR 6340 W BARRY AVE,"6340 W BARRY AVE, CHICAGO IL 60634"
104,Block Level LSLR,PUBLIC,3101 N NARRAGANSETT AVE,Completed,60634,30,2025-09-02 14:52:00,2025.0,09/2025,09-02,3101 N NARRAGANSETT AVE,567,Block Level LSLR 3101 N NARRAGANSETT AVE,"3101 N NARRAGANSETT AVE, CHICAGO IL 60634"


In [234]:
display(step6)

,id,address,city,state,zipcode,geocoded_address,is_match,is_exact,returned_address,coordinates,tiger_line,side,state_fips,county_fips,tract,block,longitude,latitude
0,1,6011 S CAMPBELL AVE,Chicago,IL,60629,"6011 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6011 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686151139103,41.784467129116",111804823,L,17,31,835000,1010,-87.686151,41.784467
1,2,6011 S CAMPBELL AVE,Chicago,IL,60629,"6011 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6011 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686151139103,41.784467129116",111804823,L,17,31,835000,1010,-87.686151,41.784467
2,3,2500 W 60TH ST,Chicago,IL,60629,"2500 W 60TH ST, Chicago, IL, 60629",Match,Exact,"2500 W 60TH ST, CHICAGO, IL, 60629","-87.686310582357,41.78475446124",605308184,R,17,31,835000,1006,-87.686311,41.784754
3,4,2500 W 60TH ST,Chicago,IL,60629,"2500 W 60TH ST, Chicago, IL, 60629",Match,Exact,"2500 W 60TH ST, CHICAGO, IL, 60629","-87.686310582357,41.78475446124",605308184,R,17,31,835000,1006,-87.686311,41.784754
4,5,6000 S CAMPBELL AVE,Chicago,IL,60629,"6000 S CAMPBELL AVE, Chicago, IL, 60629",Match,Exact,"6000 S CAMPBELL AVE, CHICAGO, IL, 60629","-87.686309136878,41.784638811931",111804823,R,17,31,835000,1009,-87.686309,41.784639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
551,564,6330 W BARRY AVE,Chicago,IL,60634,"6330 W BARRY AVE, Chicago, IL, 60634",Match,Exact,"6330 W BARRY AVE, CHICAGO, IL, 60634","-87.784557624546,41.936564155292",605161703,R,17,31,190401,1014,-87.784558,41.936564
552,565,6334 W BARRY AVE,Chicago,IL,60634,"6334 W BARRY AVE, Chicago, IL, 60634",Match,Exact,"6334 W BARRY AVE, CHICAGO, IL, 60634","-87.784697038734,41.936562367938",605161703,R,17,31,190401,1014,-87.784697,41.936562
553,566,6340 W BARRY AVE,Chicago,IL,60634,"6340 W BARRY AVE, Chicago, IL, 60634",Match,Exact,"6340 W BARRY AVE, CHICAGO, IL, 60634","-87.784906160015,41.936559686907",605161703,R,17,31,190401,1014,-87.784906,41.936560
554,567,3101 N NARRAGANSETT AVE,Chicago,IL,60634,"3101 N NARRAGANSETT AVE, Chicago, IL, 60634",Match,Exact,"3101 N NARRAGANSETT AVE, CHICAGO, IL, 60634","-87.785797121722,41.936549496226",111750757,R,17,31,190401,1013,-87.785797,41.936549


In [235]:
step6_trunc = step6
step6_trunc = step6_trunc[['address','geocoded_address','returned_address','tiger_line','tract','longitude','latitude']]

In [236]:
print(len(step5))
print(len(step5.columns))
print(len(step6))
print(len(step6.columns))


562
14
556
18


In [237]:
# ugh ok lemme try something

step5['full_address'] = step5['full_address'].replace("CHICAGO IL","CHICAGO, IL", regex=True)
display(step5)

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,clean_address,id,program_clean_address,full_address
482,Block Level LSLR,CONSUMER,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:13:00,2025.0,12/2025,12-19,6011 S CAMPBELL AVE,1,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO, IL 60629"
483,Block Level LSLR,PUBLIC,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:10:00,2025.0,12/2025,12-19,6011 S CAMPBELL AVE,2,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO, IL 60629"
71,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 W 60TH ST,3,Block Level LSLR 2500 W 60TH ST,"2500 W 60TH ST, CHICAGO, IL 60629"
72,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,2500 W 60TH ST,4,Block Level LSLR 2500 W 60TH ST,"2500 W 60TH ST, CHICAGO, IL 60629"
474,Block Level LSLR,CONSUMER,6000 S CAMPBELL AVE,Completed,60629,16,2025-12-18 00:00:00,2025.0,12/2025,12-18,6000 S CAMPBELL AVE,5,Block Level LSLR 6000 S CAMPBELL AVE,"6000 S CAMPBELL AVE, CHICAGO, IL 60629"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
553,Block Level LSLR,CONSUMER,6330 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,6330 W BARRY AVE,564,Block Level LSLR 6330 W BARRY AVE,"6330 W BARRY AVE, CHICAGO, IL 60634"
555,Block Level LSLR,CONSUMER,6334 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,6334 W BARRY AVE,565,Block Level LSLR 6334 W BARRY AVE,"6334 W BARRY AVE, CHICAGO, IL 60634"
559,Block Level LSLR,CONSUMER,6340 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,6340 W BARRY AVE,566,Block Level LSLR 6340 W BARRY AVE,"6340 W BARRY AVE, CHICAGO, IL 60634"
104,Block Level LSLR,PUBLIC,3101 N NARRAGANSETT AVE,Completed,60634,30,2025-09-02 14:52:00,2025.0,09/2025,09-02,3101 N NARRAGANSETT AVE,567,Block Level LSLR 3101 N NARRAGANSETT AVE,"3101 N NARRAGANSETT AVE, CHICAGO, IL 60634"


In [238]:
step6_trunc

,address,geocoded_address,returned_address,tiger_line,tract,longitude,latitude
0,6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, Chicago, IL, 60629","6011 S CAMPBELL AVE, CHICAGO, IL, 60629",111804823,835000,-87.686151,41.784467
1,6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, Chicago, IL, 60629","6011 S CAMPBELL AVE, CHICAGO, IL, 60629",111804823,835000,-87.686151,41.784467
2,2500 W 60TH ST,"2500 W 60TH ST, Chicago, IL, 60629","2500 W 60TH ST, CHICAGO, IL, 60629",605308184,835000,-87.686311,41.784754
3,2500 W 60TH ST,"2500 W 60TH ST, Chicago, IL, 60629","2500 W 60TH ST, CHICAGO, IL, 60629",605308184,835000,-87.686311,41.784754
4,6000 S CAMPBELL AVE,"6000 S CAMPBELL AVE, Chicago, IL, 60629","6000 S CAMPBELL AVE, CHICAGO, IL, 60629",111804823,835000,-87.686309,41.784639
...,...,...,...,...,...,...,...
551,6330 W BARRY AVE,"6330 W BARRY AVE, Chicago, IL, 60634","6330 W BARRY AVE, CHICAGO, IL, 60634",605161703,190401,-87.784558,41.936564
552,6334 W BARRY AVE,"6334 W BARRY AVE, Chicago, IL, 60634","6334 W BARRY AVE, CHICAGO, IL, 60634",605161703,190401,-87.784697,41.936562
553,6340 W BARRY AVE,"6340 W BARRY AVE, Chicago, IL, 60634","6340 W BARRY AVE, CHICAGO, IL, 60634",605161703,190401,-87.784906,41.936560
554,3101 N NARRAGANSETT AVE,"3101 N NARRAGANSETT AVE, Chicago, IL, 60634","3101 N NARRAGANSETT AVE, CHICAGO, IL, 60634",111750757,190401,-87.785797,41.936549


In [239]:
join2 = pd.merge(step5, step6_trunc, left_on='full_address', right_on='returned_address', how='left')

print(len(join2))
print(len(join2.columns))

562
21


In [240]:
display(join2)

,program,Public or Consumer,Work Address,Status,ZIP,Ward,Date Completed,completed_year,completed_month_year,completed_day,...,id,program_clean_address,full_address,address,geocoded_address,returned_address,tiger_line,tract,longitude,latitude
0,Block Level LSLR,CONSUMER,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:13:00,2025.0,12/2025,12-19,...,1,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO, IL 60629",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Block Level LSLR,PUBLIC,6011 S CAMPBELL AVE,Completed,60629,16,2025-12-19 12:10:00,2025.0,12/2025,12-19,...,2,Block Level LSLR 6011 S CAMPBELL AVE,"6011 S CAMPBELL AVE, CHICAGO, IL 60629",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Block Level LSLR,CONSUMER,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,...,3,Block Level LSLR 2500 W 60TH ST,"2500 W 60TH ST, CHICAGO, IL 60629",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Block Level LSLR,PUBLIC,2500 10 W 60TH ST,Completed,60629,16,2025-12-19 00:00:00,2025.0,12/2025,12-19,...,4,Block Level LSLR 2500 W 60TH ST,"2500 W 60TH ST, CHICAGO, IL 60629",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Block Level LSLR,CONSUMER,6000 S CAMPBELL AVE,Completed,60629,16,2025-12-18 00:00:00,2025.0,12/2025,12-18,...,5,Block Level LSLR 6000 S CAMPBELL AVE,"6000 S CAMPBELL AVE, CHICAGO, IL 60629",NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,Block Level LSLR,CONSUMER,6330 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,564,Block Level LSLR 6330 W BARRY AVE,"6330 W BARRY AVE, CHICAGO, IL 60634",NaN,NaN,NaN,NaN,NaN,NaN,NaN
558,Block Level LSLR,CONSUMER,6334 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,565,Block Level LSLR 6334 W BARRY AVE,"6334 W BARRY AVE, CHICAGO, IL 60634",NaN,NaN,NaN,NaN,NaN,NaN,NaN
559,Block Level LSLR,CONSUMER,6340 W BARRY AVE,Completed,60634,30,2025-09-03 00:00:00,2025.0,09/2025,09-03,...,566,Block Level LSLR 6340 W BARRY AVE,"6340 W BARRY AVE, CHICAGO, IL 60634",NaN,NaN,NaN,NaN,NaN,NaN,NaN
560,Block Level LSLR,PUBLIC,3101 N NARRAGANSETT AVE,Completed,60634,30,2025-09-02 14:52:00,2025.0,09/2025,09-02,...,567,Block Level LSLR 3101 N NARRAGANSETT AVE,"3101 N NARRAGANSETT AVE, CHICAGO, IL 60634",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [241]:
# okay I'm taking this out then bringing it back in because this is cwazy

In [242]:
step5.to_csv('./step5_interim.csv')
step6.to_csv('./step6_interim.csv')
step6_trunc.to_csv('./step6trunc_interim.csv')

In [243]:
#### BREAK TO BRING IT INTO SHEETS AND JOIN THIS AND THEN BRING BACK IN

In [244]:
# joined file from sheets #

joined = pd.read_csv('./TKTKTKTKT.csv')
step6 = joined

FileNotFoundError: [Errno 2] No such file or directory: './TKTKTKTKT.csv'

In [ ]:
# standardize daycare program
step6['program'] = np.where(step6['program'] == 'Daycare Private', 'Daycare', step6['program'])
step6['program'] = np.where(step6['program'] == 'Daycare Public', 'Daycare', step6['program'])

# combine sewer and water main
step6['program'] = np.where(step6['program'] == 'CIP Sewer Main', 'CIP Sewer and Water Main', step6['program'])
step6['program'] = np.where(step6['program'] == 'CIP Water Main', 'CIP Sewer and Water Main', step6['program'])

KeyError: 'program'

In [ ]:
existing = []
# can't do this yet. have to wait until clean because you don't have the base files :( )